In [1]:
import os
import certifi 
import requests
from dotenv import load_dotenv

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.tools import tool
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain import hub

/home/rare/miniconda3/envs/langagent/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain.agents import create_react_agent, AgentExecutor

In [3]:
load_dotenv()

GOOGLE_API_KEY = os.getenv('GOOGLE_API_KEY')
TAVILY_API_KEY = os.getenv('TAVILY_API_KEY')

In [4]:
search_tool = TavilySearchResults(max_results=2)

In [5]:
result=search_tool.invoke('Give  me the latest news on AI')
result 

[{'url': 'https://www.artificialintelligence-news.com',
  'content': 'Cybersecurity AI\n\nAugust 18, 2026\n\n### Okta targets AI agent token costs with MCP scoping\n\nData Engineering & MLOps\n\nAugust 13, 2026\n\n### Red Hat, NVIDIA, IBM back project turning AI policy into code\n\nGovernance, Regulation & Policy\n\nAugust 4, 2026\n\n#### Industries\n\n### NVIDIA Jetson Orin Nano 2 brings physical AI to drones and robots\n\nPhysical AI\n\nAugust 26, 2026\n\n### HoneyBook bets on agentic AI to streamline small business operations with its new Claude connector\n\nWorld of Work\n\nAugust 6, 2026\n\n### Zuckerberg details Meta’s personal AI superintelligence strategy\n\nAI and Us\n\nJuly 30, 2026\n\n#### Deep Learning\n\n### Aviva deploys AI to stop £230M in sophisticated insurance fraud\n\nAI in Action\n\nJune 8, 2026 [...] AI in Action\n\nJuly 29, 2025\n\n#### Computer Vision\n\n### Microsoft’s Majorana 2 quantum chip is also a case study for agentic AI in R&D\n\nInside AI\n\nJune 3, 202

In [7]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0,
)

In [9]:
response = llm.invoke("Tell me a joke about AI")
response

AIMessage(content='How many AIs does it take to change a lightbulb?\n\nNone. It just stands in the dark and confidently convinces you the light is already on!', response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'safety_ratings': []}, id='run-525fc76c-12c8-4b1e-bc7b-d3158d7c9e00-0')

## ReAct research agent

`handle_parsing_errors` sends malformed model output back to the agent with an explicit correction instead of raising `OutputParserException`.

In [10]:
tools = [search_tool]
prompt = hub.pull("hwchase17/react")
agent = create_react_agent(llm, tools, prompt)

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=(
        "Invalid ReAct format. Follow the format in the prompt exactly. "
        "If you already know the answer, respond with 'Final Answer: <answer>'."
    ),
    max_iterations=5,
)

/home/rare/miniconda3/envs/langagent/lib/python3.11/site-packages/langchain/hub.py:86: DeprecationWarning: The `langchainhub sdk` is deprecated.
Please use the `langsmith sdk` instead:
  pip install langsmith
Use the `pull_prompt` method.
  res_dict = client.pull_repo(owner_repo_commit)


In [11]:
response = agent_executor.invoke({
    "input": "Tell me the latest news about Iran and the USA"
})

print(response["output"])



> Entering new AgentExecutor chain...
Action: tavily_search_results_json
Action Input: latest news Iran USA relations conflicts 2024 2025[{'url': 'https://en.wikipedia.org/wiki/2025%E2%80%932026_Iran%E2%80%93United_States_negotiations', 'content': '## Trump\'s letter to Khamenei\n\nOn March 7, 2025, Trump announced that he had written a letter to Ali Khamenei, expressing his desire to initiate new nuclear negotiations with Iran. He warned that failure to accept the proposal could expose Iran to serious military consequences. Trump also imposed additional sanctions on the country and pledged to reduce Iran\'s oil exports to zero. According to Emirati political scientist Abdulkhaleq Abdullah, the letter included demands that Iran fully dismantle its nuclear program, halt all enrichment, and end support for regional proxy groups, with compliance required within two months. In exchange, the US offered to lift sanctions and normalize relations, while warning that failure to negotiate woul